# Weibull Survival Analysis — Customer LTV & Opportunity Cost
### Dataset: Olist Brazilian E-Commerce (Kaggle)

---

> **Central question:** Is customer segmentation a data problem or a marketing problem?
>
> This notebook uses Weibull survival analysis to bridge **past behavior → future probability → economic decision**.

---

## Section 0 — Setup & Data Loading

Download the dataset from [Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data) and place all CSVs inside a folder named `olist_data/` in the same directory as this notebook.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import weibull_min
from scipy.special import gamma as gamma_fn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

# ── Plot style ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : '#F8FAFC',
    'axes.facecolor'   : '#F8FAFC',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'grid.linestyle'   : '--',
    'font.family'      : 'DejaVu Sans',
})

PALETTE = {
    'Champions' : '#10B981',
    'Nurture'   : '#3B82F6',
    'At Risk'   : '#F59E0B',
    'Dormant'   : '#94A3B8',
    'accent'    : '#EF4444',
    'dark'      : '#1E293B',
}

PATH = './olist_data/'

orders    = pd.read_csv(PATH + 'olist_orders_dataset.csv')
items     = pd.read_csv(PATH + 'olist_order_items_dataset.csv')
customers = pd.read_csv(PATH + 'olist_customers_dataset.csv')
payments  = pd.read_csv(PATH + 'olist_order_payments_dataset.csv')

print('Tables loaded:')
for name, df in [('orders', orders), ('items', items),
                 ('customers', customers), ('payments', payments)]:
    print(f'  {name:12s}: {len(df):>7,} rows  |  cols: {list(df.columns)}')

---
## Section 1 — Feature Engineering: `media_dias`

The key variable is the **average inter-purchase interval** per customer (called `media_dias`).
It quantifies purchase frequency: the smaller it is, the more loyal and valuable the customer.

**Steps:**
1. Keep only delivered orders
2. Join with `customer_unique_id` (Olist uses a different ID per order — this is the true customer key)
3. Sort purchases chronologically per customer
4. Compute the difference in days between consecutive purchases
5. Average those differences → `media_dias`

In [ ]:
# ── Filter & parse ───────────────────────────────────────────
orders = orders[orders['order_status'] == 'delivered'].copy()
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

# Revenue per order (sum of payment values)
revenue = (
    payments[payments['payment_type'] != 'not_defined']
    .groupby('order_id')['payment_value']
    .sum()
    .reset_index()
    .rename(columns={'payment_value': 'order_value'})
)

# Merge everything
df = (
    orders[['order_id', 'customer_id', 'order_purchase_timestamp']]
    .merge(customers[['customer_id', 'customer_unique_id']], on='customer_id')
    .merge(revenue, on='order_id')
    .rename(columns={
        'customer_unique_id'      : 'customer',
        'order_purchase_timestamp': 'purchase_date',
    })
)

# ── Inter-purchase interval ──────────────────────────────────
df = df.sort_values(['customer', 'purchase_date'])
df['interval_days'] = df.groupby('customer')['purchase_date'].diff().dt.days

# Only repeat buyers (need at least 2 purchases to compute an interval)
repeat_buyers = (
    df.dropna(subset=['interval_days'])
    .groupby('customer')
    .agg(
        media_dias  = ('interval_days', 'mean'),
        n_purchases = ('order_id',      'count'),
        avg_ticket  = ('order_value',   'mean'),
        total_spent = ('order_value',   'sum'),
    )
    .reset_index()
)
repeat_buyers = repeat_buyers[repeat_buyers['media_dias'] > 0]

print(f'Total delivered orders : {len(df):,}')
print(f'Unique customers       : {df["customer"].nunique():,}')
print(f'Repeat buyers (2+)     : {len(repeat_buyers):,}')
print()
print('media_dias distribution:')
print(repeat_buyers['media_dias'].describe().round(1).to_string())

---
## Section 2 — The Weibull Distribution: Mathematical Derivation

### 2.1 Why Weibull?

The exponential distribution (used in basic survival models) assumes a **constant hazard rate** — 
i.e., the probability of repurchase per unit of time never changes regardless of how long the customer has been silent. 
This is rarely true in retail.

The **Weibull distribution** generalizes this with a shape parameter $\beta$ that allows the hazard to increase, decrease, or remain constant over time.

---

### 2.2 Probability Density Function (PDF)

$$
f(t; \beta, \lambda) = \frac{\beta}{\lambda} \left(\frac{t}{\lambda}\right)^{\beta - 1} \exp\left[-\left(\frac{t}{\lambda}\right)^{\beta}\right], \quad t > 0
$$

- $\beta$ (shape / **forma**): controls how the hazard evolves over time
- $\lambda$ (scale / **escala**): stretches or compresses the distribution along the time axis

---

### 2.3 Cumulative Distribution Function (CDF)

The CDF gives the probability that repurchase happens **before or at time $t$**:

$$
F(t) = 1 - \exp\left[-\left(\frac{t}{\lambda}\right)^{\beta}\right]
$$

---

### 2.4 Survival Function

The survival function $S(t)$ is the complement of the CDF — the probability the customer has **not yet repurchased** by time $t$:

$$
S(t) = 1 - F(t) = \exp\left[-\left(\frac{t}{\lambda}\right)^{\beta}\right]
$$

---

### 2.5 Hazard Function

The hazard function $h(t)$ is the **instantaneous rate of repurchase** at time $t$, given that the customer hasn't repurchased yet:

$$
h(t) = \frac{f(t)}{S(t)} = \frac{\beta}{\lambda} \left(\frac{t}{\lambda}\right)^{\beta - 1}
$$

**Interpretation of $\beta$:**

| $\beta$ | Hazard shape | Retail meaning |
|---|---|---|
| $\beta < 1$ | Decreasing | Customers who don't buy quickly become increasingly unlikely to return |
| $\beta = 1$ | Constant | Memoryless — repurchase probability doesn't change with time (exponential) |
| $\beta > 1$ | Increasing | Customers become more likely to repurchase the longer they wait |

---

### 2.6 Expected Value

$$
\mathbb{E}[T] = \lambda \cdot \Gamma\!\left(1 + \frac{1}{\beta}\right)
$$

where $\Gamma$ is the Gamma function. This is the **theoretical mean inter-purchase interval**.

---

### 2.7 Maximum Likelihood Estimation (MLE)

Given $n$ observed inter-purchase intervals $t_1, t_2, \ldots, t_n$, we want to find $\hat{\beta}$ and $\hat{\lambda}$ that maximize the log-likelihood:

$$
\ell(\beta, \lambda) = n \ln \beta - n \beta \ln \lambda + (\beta - 1) \sum_{i=1}^n \ln t_i - \sum_{i=1}^n \left(\frac{t_i}{\lambda}\right)^{\beta}
$$

Taking the partial derivative with respect to $\lambda$ and setting to zero gives the closed-form solution for $\hat{\lambda}$:

$$
\hat{\lambda} = \left(\frac{1}{n} \sum_{i=1}^n t_i^{\hat{\beta}}\right)^{1/\hat{\beta}}
$$

The estimate $\hat{\beta}$ is obtained numerically (no closed form) — `scipy.stats.weibull_min.fit()` handles this via numerical optimization.

---

> **Intuitive summary:** We're asking — *what shape of "time-to-repurchase" distribution best explains what we observed?* MLE finds the $\beta$ and $\lambda$ that make our observed data most probable under the Weibull model.

In [ ]:
# ── Fit Weibull via MLE ──────────────────────────────────────
x = repeat_buyers['media_dias'].values

# floc=0 fixes the location parameter to 0 (no shift — intervals start at 0)
shape, loc, scale = weibull_min.fit(x, floc=0)

theoretical_mean = scale * gamma_fn(1 + 1 / shape)

print('── Weibull Parameters (MLE) ──────────────────────')
print(f'β (shape) : {shape:.4f}')
print(f'λ (scale) : {scale:.2f} days')
print(f'E[T]      : {theoretical_mean:.1f} days  (theoretical mean interval)')
print(f'Observed mean: {x.mean():.1f} days')
print()

if shape < 1:
    msg = f'β = {shape:.2f} < 1 → DECREASING hazard.\nCustomers who repurchase, do so early. The longer the silence, the lower the chance of return.'
elif abs(shape - 1) < 0.05:
    msg = f'β ≈ 1 → CONSTANT hazard. Memoryless process (exponential).'
else:
    msg = f'β = {shape:.2f} > 1 → INCREASING hazard. Customers grow more likely to repurchase over time.'
print('Interpretation:', msg)

---
### Plot 1 — Weibull PDF Fit vs. Observed Data

**What to look for:** The green curve (fitted Weibull) should follow the shape of the histogram.
A good fit here validates that the Weibull model is appropriate for this customer base.
The right-skewed shape is typical in retail: most repeat purchases happen within the first 60 days,
but there's a long tail of infrequent buyers.

In [ ]:
x_clip = np.percentile(x, 98)
x_plot = np.linspace(0.1, x_clip, 600)
pdf_v  = weibull_min.pdf(x_plot, shape, loc=0, scale=scale)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(x, bins=70, density=True, alpha=0.35, color=PALETTE['dark'],
        label='Observed data (repeat buyers)', range=(0, x_clip))
ax.plot(x_plot, pdf_v, color=PALETTE['Champions'], lw=2.5,
        label=f'Weibull MLE fit   β={shape:.2f}, λ={scale:.0f} days')
ax.axvline(theoretical_mean, color=PALETTE['accent'], lw=1.5, ls='--',
           label=f'E[T] = {theoretical_mean:.0f} days')
ax.set_xlabel('Average inter-purchase interval (days)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Weibull Distribution Fit on Olist Repeat Buyers', fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('plot1_weibull_fit.png', dpi=150, bbox_inches='tight')
plt.show()

---
### Plot 2 — Survival Function & Hazard Function

**Survival $S(t)$:** The fraction of customers who have **not yet repurchased** by day $t$.
A steep initial drop means most loyal customers return quickly.

**Hazard $h(t)$:** The instantaneous repurchase rate at day $t$ *conditional on not having repurchased yet*.
With $\beta < 1$, this curve is strictly decreasing — confirming that **the first weeks after a purchase are the highest-value window for CRM action**.

In [ ]:
surv_v   = weibull_min.sf(x_plot, shape, loc=0, scale=scale)
hazard_v = weibull_min.pdf(x_plot, shape, loc=0, scale=scale) / (surv_v + 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# ── Survival ──
ax = axes[0]
ax.plot(x_plot, surv_v, color=PALETTE['Nurture'], lw=2.5)
ax.fill_between(x_plot, surv_v, alpha=0.12, color=PALETTE['Nurture'])
for t_mark, col in [(30, PALETTE['Champions']), (90, PALETTE['At Risk']), (180, PALETTE['accent'])]:
    s = weibull_min.sf(t_mark, shape, loc=0, scale=scale)
    ax.axvline(t_mark, color=col, ls=':', lw=1.3)
    ax.annotate(f'{t_mark}d: {s:.0%} still inactive',
                xy=(t_mark, s), xytext=(t_mark + 8, s + 0.03),
                fontsize=8, color=col)
ax.set_xlabel('Days since last purchase', fontsize=11)
ax.set_ylabel('S(t) — Fraction not yet repurchased', fontsize=11)
ax.set_title('Survival Function $S(t)$', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# ── Hazard ──
ax = axes[1]
ax.plot(x_plot, hazard_v, color=PALETTE['At Risk'], lw=2.5)
ax.fill_between(x_plot, hazard_v, alpha=0.12, color=PALETTE['At Risk'])
ax.set_xlabel('Days since last purchase', fontsize=11)
ax.set_ylabel('h(t) — Instantaneous repurchase rate', fontsize=11)
ax.set_title(f'Hazard Function $h(t)$   [β={shape:.2f} → decreasing]', fontsize=12, fontweight='bold')
ax.annotate('Window of highest\nrepurchase probability',
            xy=(10, hazard_v[5]), xytext=(60, hazard_v[5] * 1.4),
            fontsize=8.5, color=PALETTE['dark'],
            arrowprops=dict(arrowstyle='->', color=PALETTE['dark']))

plt.suptitle('Survival & Hazard Functions — Weibull Model', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot2_survival_hazard.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3 — The Bridge: Past → Future

### 3.1 Conditional Repurchase Probability

The key question for CRM is not *"what is the unconditional probability of repurchase?"* but:

> *Given that a customer has already been silent for $t_0$ days — what is the probability they repurchase within the next $\Delta t$ days?*

This is a conditional probability derived directly from the Weibull CDF:

$$
P(T \leq t_0 + \Delta t \mid T > t_0) = \frac{F(t_0 + \Delta t) - F(t_0)}{1 - F(t_0)} = \frac{F(t_0 + \Delta t) - F(t_0)}{S(t_0)}
$$

This is the **bridge between the fitted model and today's decision**.
Each customer enters the formula with their own $t_0$ (days since last purchase),
yielding a personalized repurchase probability for the next 30 days.

> **Intuitive summary:** Two customers can have identical purchase history but be in completely different states today. A customer silent for 5 days is fundamentally different from one silent for 200 days — this formula captures that difference precisely.

In [ ]:
def conditional_prob(t0, delta_t, shape, scale):
    """
    P(repurchase within delta_t days | already t0 days silent)
    Uses Weibull conditional survival.
    """
    F_t0        = weibull_min.cdf(t0,            shape, loc=0, scale=scale)
    F_t0_delta  = weibull_min.cdf(t0 + delta_t,  shape, loc=0, scale=scale)
    S_t0        = 1 - F_t0
    with np.errstate(divide='ignore', invalid='ignore'):
        prob = np.where(S_t0 > 1e-6, (F_t0_delta - F_t0) / S_t0, 0.0)
    return prob

HORIZON = 30  # next 30 days

# Show the decay table
print(f'P(repurchase within {HORIZON}d | t0 days silent)\n')
print(f'{"Days silent":>14}  {"P(repurchase)":>14}')
print('-' * 32)
for t0 in [7, 14, 30, 60, 90, 120, 180, 270, 365]:
    p = conditional_prob(t0, HORIZON, shape, scale)
    print(f'{t0:>14}  {p:>13.1%}')

---
### Plot 3 — Conditional Probability Decay Curve

**What to look for:** How fast the probability collapses as days of silence accumulate.
The steeper the curve in the first 30–60 days, the more **time-sensitive** your CRM actions need to be.
This curve is the direct output of the Weibull model applied to the current state of each customer.

In [ ]:
t0_range = np.linspace(1, 500, 500)
p_curve  = conditional_prob(t0_range, HORIZON, shape, scale)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(t0_range, p_curve, color=PALETTE['dark'], lw=2.5)
ax.fill_between(t0_range, p_curve, alpha=0.08, color=PALETTE['dark'])

annotations = [
    (30,  PALETTE['Champions']),
    (60,  PALETTE['Nurture']),
    (90,  PALETTE['At Risk']),
    (180, PALETTE['accent']),
    (365, PALETTE['Dormant']),
]
for days, color in annotations:
    p = conditional_prob(days, HORIZON, shape, scale)
    ax.scatter(days, p, color=color, zorder=5, s=50)
    ax.annotate(f'{days}d → {p:.0%}',
                xy=(days, p), xytext=(days + 12, p + 0.015),
                fontsize=8.5, color=color,
                arrowprops=dict(arrowstyle='->', color=color, lw=0.8))

ax.set_xlabel('Days since last purchase ($t_0$)', fontsize=11)
ax.set_ylabel(f'P(repurchase within {HORIZON}d | $t_0$)', fontsize=11)
ax.set_title('Conditional Repurchase Probability — The Bridge Past → Future', fontsize=13, fontweight='bold', pad=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.tight_layout()
plt.savefig('plot3_conditional_prob.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 4 — Opportunity Cost Framework

### 4.1 Expected Revenue per Customer

For each customer $i$ with days-of-silence $t_{0,i}$ and average ticket $\bar{v}_i$:

$$
\mathbb{E}[\text{Revenue}_i] = P(T_i \leq t_{0,i} + \Delta t \mid T_i > t_{0,i}) \times \bar{v}_i
$$

### 4.2 Expected Net Value (after activation cost)

$$
\text{ENL}_i = \mathbb{E}[\text{Revenue}_i] \times m - C_a
$$

where $m$ is the profit margin and $C_a$ is the cost of reaching the customer (email, SMS, push notification).

### 4.3 Opportunity Cost of Inaction

If you do **nothing**, the expected margin left on the table per customer is:

$$
\text{OC}_i = \mathbb{E}[\text{Revenue}_i] \times m
$$

### 4.4 ROI of Activation

$$
\text{ROI}_i = \frac{\mathbb{E}[\text{Revenue}_i] \times m - C_a}{C_a}
$$

A customer is worth activating if and only if $\text{ROI}_i > 0$, i.e., $\mathbb{E}[\text{Revenue}_i] \times m > C_a$.

> **Intuitive summary:** The opportunity cost is not abstract — it's the margin you're leaving on the table by not communicating with a specific customer today. The model makes this number calculable, per customer, in real currency.

In [ ]:
MARGIN          = 0.35   # 35% net margin (adjust to your reality)
ACTIVATION_COST = 8.0    # R$ cost to reach one customer

cutoff_date = df['purchase_date'].max()

# Current state of each customer
customer_state = (
    df.groupby('customer')
    .agg(
        last_purchase = ('purchase_date', 'max'),
        avg_ticket    = ('order_value',   'mean'),
        n_orders      = ('order_id',      'count'),
        total_spent   = ('order_value',   'sum'),
    )
    .reset_index()
)
customer_state['days_silent'] = (cutoff_date - customer_state['last_purchase']).dt.days
customer_state = customer_state.merge(
    repeat_buyers[['customer', 'media_dias']], on='customer', how='inner'
)

# Core calculations
customer_state['prob_30d'] = conditional_prob(
    customer_state['days_silent'].values, HORIZON, shape, scale
)
customer_state['expected_revenue']   = customer_state['prob_30d'] * customer_state['avg_ticket']
customer_state['expected_net_value'] = customer_state['expected_revenue'] * MARGIN - ACTIVATION_COST
customer_state['opportunity_cost']   = (customer_state['expected_revenue'] * MARGIN).clip(lower=0)
customer_state['roi_activation']     = (
    (customer_state['expected_revenue'] * MARGIN - ACTIVATION_COST) / ACTIVATION_COST
).replace([np.inf, -np.inf], np.nan)

total_oc         = customer_state['opportunity_cost'].sum()
positive_roi_pct = (customer_state['roi_activation'] > 0).mean()

print(f'Customers in scoring base        : {len(customer_state):,}')
print(f'Total 30d opportunity cost       : R${total_oc:,.0f}')
print(f'Customers with positive ROI      : {positive_roi_pct:.1%}')
print(f'Avg opportunity cost per customer: R${customer_state["opportunity_cost"].mean():.2f}')

---
## Section 5 — Segmentation

### 5.1 The 2×2 Matrix Logic

We split customers along two economically meaningful axes:

- **X-axis:** $P(\text{repurchase in 30d})$ — the **urgency** axis (model output)
- **Y-axis:** Average ticket — the **value** axis (observed data)

The median of each axis becomes the threshold, producing four segments with distinct **optimal actions**:

| Segment | Probability | Ticket | Strategy |
|---|---|---|---|
| **Champions** | High | High | Loyalty program, upsell, protect |
| **At Risk** | Low | High | Urgent reactivation offer — highest OC |
| **Nurture** | High | Low | Cross-sell to grow basket size |
| **Dormant** | Low | Low | Low-cost drip or reallocate budget |

In [ ]:
prob_med   = customer_state['prob_30d'].median()
ticket_med = customer_state['avg_ticket'].median()

def segment(row):
    hi_prob   = row['prob_30d']   >= prob_med
    hi_ticket = row['avg_ticket'] >= ticket_med
    if hi_prob and hi_ticket:       return 'Champions'
    elif hi_prob and not hi_ticket: return 'Nurture'
    elif not hi_prob and hi_ticket: return 'At Risk'
    else:                           return 'Dormant'

customer_state['segment'] = customer_state.apply(segment, axis=1)

summary = (
    customer_state.groupby('segment')
    .agg(
        n_customers    = ('customer',          'count'),
        avg_prob_30d   = ('prob_30d',          'mean'),
        avg_ticket_R$  = ('avg_ticket',        'mean'),
        total_opp_cost = ('opportunity_cost',  'sum'),
        avg_roi        = ('roi_activation',    'mean'),
    )
    .round(2)
    .sort_values('total_opp_cost', ascending=False)
)
print('── Segment Summary ──────────────────────────────────────────')
print(summary.to_string())
print(f'\nMedian prob threshold : {prob_med:.3f}')
print(f'Median ticket threshold: R${ticket_med:.2f}')

---
### Plot 4 — CRM Prioritization Matrix (Scatter)

**What to look for:**
- **At Risk** (top-left): These are your most expensive silent customers. High ticket, low probability of spontaneous return. The opportunity cost is highest here.
- **Champions** (top-right): Protect and invest. High ROI on activation.
- The dashed lines are median thresholds — the segmentation boundary.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sample = customer_state.sample(min(1200, len(customer_state)), random_state=42)
for seg in ['Champions', 'Nurture', 'At Risk', 'Dormant']:
    sub = sample[sample['segment'] == seg]
    ax.scatter(sub['prob_30d'], sub['avg_ticket'],
               color=PALETTE[seg], alpha=0.5, s=20, label=seg)

ax.axvline(prob_med,   color='gray', ls='--', lw=1.2, alpha=0.7)
ax.axhline(ticket_med, color='gray', ls='--', lw=1.2, alpha=0.7)

offset = 0.005
ax.text(prob_med * 0.15, ticket_med * 1.55, '⚠  At Risk',    fontsize=10, color=PALETTE['At Risk'],    alpha=0.9)
ax.text(prob_med * 1.05, ticket_med * 1.55, '🎯 Champions',  fontsize=10, color=PALETTE['Champions'],  alpha=0.9)
ax.text(prob_med * 0.15, ticket_med * 0.25, '💤 Dormant',    fontsize=10, color=PALETTE['Dormant'],    alpha=0.9)
ax.text(prob_med * 1.05, ticket_med * 0.25, '📢 Nurture',    fontsize=10, color=PALETTE['Nurture'],    alpha=0.9)

ax.set_xlabel('P(repurchase within 30 days)', fontsize=11)
ax.set_ylabel('Average ticket (R$)', fontsize=11)
ax.set_title('CRM Prioritization Matrix\nP(repurchase) × Average Ticket', fontsize=13, fontweight='bold', pad=12)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(fontsize=9, markerscale=2)
plt.tight_layout()
plt.savefig('plot4_crm_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
### Plot 5 — Opportunity Cost Distribution & ROI by Segment

**Left:** Distribution of opportunity cost per customer. Most customers have a small OC, but the right tail is significant — those are your priority targets.

**Right:** Boxplot of activation ROI per segment. The horizontal line at 0 is the break-even. Boxes entirely above zero are segments where **every activation pays off**. Boxes crossing zero require triage.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── OC histogram ──
ax = axes[0]
oc = customer_state['opportunity_cost'].clip(
    upper=np.percentile(customer_state['opportunity_cost'], 97))
ax.hist(oc, bins=55, color=PALETTE['At Risk'], alpha=0.7, edgecolor='white')
ax.axvline(oc.mean(), color=PALETTE['accent'], lw=2, ls='--',
           label=f'Mean: R${oc.mean():.1f}')
ax.axvline(ACTIVATION_COST, color=PALETTE['Champions'], lw=1.5, ls=':',
           label=f'Activation cost: R${ACTIVATION_COST:.0f}')
ax.set_xlabel('Opportunity cost per customer (R$)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Opportunity Cost Distribution\n(Cost of Doing Nothing)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

# ── ROI boxplot ──
ax = axes[1]
seg_order = ['Champions', 'Nurture', 'At Risk', 'Dormant']
roi_data  = [customer_state[customer_state['segment'] == s]['roi_activation'].dropna().values
             for s in seg_order]
bp = ax.boxplot(roi_data, patch_artist=True, notch=False,
                medianprops=dict(color='white', lw=2.5))
for patch, seg in zip(bp['boxes'], seg_order):
    patch.set_facecolor(PALETTE[seg])
    patch.set_alpha(0.75)
ax.axhline(0, color=PALETTE['accent'], lw=1.8, ls='--', label='Break-even (ROI = 0)')
ax.set_xticklabels(seg_order, fontsize=9)
ax.set_ylabel('Activation ROI', fontsize=11)
ax.set_title('ROI of Activation per Segment\n(Where to Invest CRM Budget)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('Opportunity Cost & Activation ROI', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot5_opp_cost_roi.png', dpi=150, bbox_inches='tight')
plt.show()

---
### Plot 6 — Lorenz Curve: Concentration of Opportunity Cost

**What to look for:** How far the blue curve bows away from the diagonal (perfect equality line).
The greater the bow, the more concentrated the opportunity cost in a small fraction of customers.

This is the **Gini coefficient of CRM attention** — if 20% of customers hold 80% of the opportunity cost,
then allocating budget linearly is economically irrational.

> This plot answers: *"What is the cost of treating all customers equally?"*

In [ ]:
oc_sorted = customer_state['opportunity_cost'].sort_values(ascending=False).reset_index(drop=True)
oc_cumsum = oc_sorted.cumsum() / oc_sorted.sum()
pop_frac  = np.arange(1, len(oc_cumsum) + 1) / len(oc_cumsum)
equality  = np.linspace(0, 1, len(oc_cumsum))

# Gini coefficient
gini = 1 - 2 * np.trapz(oc_cumsum.values, pop_frac)

# Top 20% concentration
idx_20  = int(0.20 * len(pop_frac))
pct_top = oc_cumsum.values[idx_20]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(pop_frac, oc_cumsum.values, color=PALETTE['Nurture'], lw=2.5,
        label=f'Opportunity cost (Gini = {gini:.2f})')
ax.plot([0, 1], [0, 1], color='gray', ls='--', lw=1.2, label='Perfect equality')
ax.fill_between(pop_frac, oc_cumsum.values, equality, alpha=0.12, color=PALETTE['Nurture'])

ax.axvline(0.20, color=PALETTE['At Risk'], ls=':', lw=1.5)
ax.axhline(pct_top, color=PALETTE['At Risk'], ls=':', lw=1.5)
ax.scatter(0.20, pct_top, color=PALETTE['At Risk'], zorder=5, s=60)
ax.annotate(f'Top 20% of customers\n= {pct_top:.0%} of opportunity cost',
            xy=(0.20, pct_top), xytext=(0.30, pct_top - 0.18),
            fontsize=9, color=PALETTE['At Risk'],
            arrowprops=dict(arrowstyle='->', color=PALETTE['At Risk']))

ax.set_xlabel('Fraction of customers (ranked by opp. cost)', fontsize=11)
ax.set_ylabel('Cumulative share of opportunity cost', fontsize=11)
ax.set_title(f'Lorenz Curve — Concentration of Opportunity Cost\nGini = {gini:.3f}',
             fontsize=13, fontweight='bold', pad=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('plot6_lorenz.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Gini coefficient of opportunity cost: {gini:.3f}')
print(f'Top 20% customers = {pct_top:.1%} of total opportunity cost')

---
## Section 6 — Final Figure (LinkedIn Post)

All 6 plots combined in a single publication-quality figure. 
**This is your screenshot for the post.**

In [ ]:
fig = plt.figure(figsize=(20, 14), facecolor='#F8FAFC')
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.30)

# ── 1. Weibull fit ──────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor('#F8FAFC')
ax1.hist(x, bins=60, density=True, alpha=0.30, color=PALETTE['dark'],
         range=(0, x_clip))
ax1.plot(x_plot, pdf_v, color=PALETTE['Champions'], lw=2.2,
         label=f'Weibull  β={shape:.2f}, λ={scale:.0f}d')
ax1.axvline(theoretical_mean, color=PALETTE['accent'], lw=1.3, ls='--',
            label=f'E[T]={theoretical_mean:.0f}d')
ax1.set_title('① Weibull PDF Fit', fontsize=11, fontweight='bold')
ax1.set_xlabel('Avg interval (days)'); ax1.legend(fontsize=8)

# ── 2. Conditional probability ─────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor('#F8FAFC')
ax2.plot(t0_range, p_curve, color=PALETTE['dark'], lw=2.2)
ax2.fill_between(t0_range, p_curve, alpha=0.07, color=PALETTE['dark'])
for days, col in [(30, PALETTE['Champions']), (90, PALETTE['At Risk']), (180, PALETTE['accent'])]:
    p = conditional_prob(days, HORIZON, shape, scale)
    ax2.axvline(days, color=col, ls='--', lw=1.1, alpha=0.8)
    ax2.annotate(f'{days}d:{p:.0%}', xy=(days, p),
                 xytext=(days+8, p+0.012), fontsize=7.5, color=col)
ax2.set_title('② Conditional Repurchase Probability', fontsize=11, fontweight='bold')
ax2.set_xlabel('Days silent (t₀)')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# ── 3. Prioritization matrix ────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor('#F8FAFC')
for seg in ['Champions', 'Nurture', 'At Risk', 'Dormant']:
    sub = sample[sample['segment'] == seg]
    ax3.scatter(sub['prob_30d'], sub['avg_ticket'],
                color=PALETTE[seg], alpha=0.45, s=14, label=seg)
ax3.axvline(prob_med,   color='gray', ls='--', lw=1, alpha=0.6)
ax3.axhline(ticket_med, color='gray', ls='--', lw=1, alpha=0.6)
ax3.text(prob_med*0.1, ticket_med*1.5, '⚠ At Risk',   fontsize=8, color=PALETTE['At Risk'])
ax3.text(prob_med*1.1, ticket_med*1.5, '🎯 Champions', fontsize=8, color=PALETTE['Champions'])
ax3.text(prob_med*0.1, ticket_med*0.2, '💤 Dormant',   fontsize=8, color=PALETTE['Dormant'])
ax3.text(prob_med*1.1, ticket_med*0.2, '📢 Nurture',   fontsize=8, color=PALETTE['Nurture'])
ax3.set_title('③ CRM Prioritization Matrix', fontsize=11, fontweight='bold')
ax3.set_xlabel('P(repurchase 30d)')
ax3.set_ylabel('Avg ticket (R$)')
ax3.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax3.legend(fontsize=7, markerscale=2)

# ── 4. Survival & Hazard ────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.set_facecolor('#F8FAFC')
ax4.plot(x_plot, surv_v,   color=PALETTE['Nurture'],  lw=2.2, label='S(t) Survival')
ax4.plot(x_plot, hazard_v / hazard_v.max(), color=PALETTE['At Risk'], lw=2.2,
         ls='--', label='h(t) Hazard (normalized)')
ax4.set_title('④ Survival & Hazard Functions', fontsize=11, fontweight='bold')
ax4.set_xlabel('Days'); ax4.legend(fontsize=8)
ax4.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# ── 5. OC distribution ─────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
ax5.set_facecolor('#F8FAFC')
ax5.hist(oc, bins=50, color=PALETTE['At Risk'], alpha=0.7, edgecolor='white')
ax5.axvline(oc.mean(), color=PALETTE['accent'], lw=2, ls='--',
            label=f'Mean R${oc.mean():.1f}')
ax5.axvline(ACTIVATION_COST, color=PALETTE['Champions'], lw=1.5, ls=':',
            label=f'Cost R${ACTIVATION_COST:.0f}')
ax5.set_title('⑤ Opportunity Cost Distribution', fontsize=11, fontweight='bold')
ax5.set_xlabel('Opp. cost per customer (R$)'); ax5.legend(fontsize=8)

# ── 6. Lorenz ──────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
ax6.set_facecolor('#F8FAFC')
ax6.plot(pop_frac, oc_cumsum.values, color=PALETTE['Nurture'], lw=2.2,
         label=f'Lorenz curve (Gini={gini:.2f})')
ax6.plot([0, 1], [0, 1], color='gray', ls='--', lw=1)
ax6.fill_between(pop_frac, oc_cumsum.values, equality, alpha=0.12, color=PALETTE['Nurture'])
ax6.axvline(0.20, color=PALETTE['At Risk'], ls=':', lw=1.3)
ax6.axhline(pct_top, color=PALETTE['At Risk'], ls=':', lw=1.3)
ax6.annotate(f'Top 20% = {pct_top:.0%}',
             xy=(0.20, pct_top), xytext=(0.30, pct_top - 0.18),
             fontsize=8, color=PALETTE['At Risk'],
             arrowprops=dict(arrowstyle='->', color=PALETTE['At Risk']))
ax6.set_title('⑥ Lorenz Curve — Opp. Cost Concentration', fontsize=11, fontweight='bold')
ax6.set_xlabel('Fraction of customers')
ax6.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax6.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax6.legend(fontsize=8)

fig.suptitle(
    'Is Customer Segmentation a Data Problem or a Marketing Problem?\n'
    'Weibull Survival Analysis → Conditional Probability → Opportunity Cost  |  Olist Brazilian E-Commerce',
    fontsize=13, fontweight='bold', y=1.01
)

plt.savefig('weibull_olist_final.png', dpi=180, bbox_inches='tight', facecolor='#F8FAFC')
plt.show()
print('Saved: weibull_olist_final.png')

---
## Section 7 — Final Decision Table

A clean summary table mapping each segment to its economic metrics and recommended CRM action.

In [ ]:
actions = {
    'Champions' : 'Loyalty program + upsell',
    'At Risk'   : 'Urgent win-back offer (30d window)',
    'Nurture'   : 'Cross-sell to grow basket size',
    'Dormant'   : 'Low-cost drip or reallocate budget',
}

decision_table = (
    customer_state.groupby('segment')
    .agg(
        customers      = ('customer',         'count'),
        avg_prob_30d   = ('prob_30d',         'mean'),
        avg_ticket_R$  = ('avg_ticket',       'mean'),
        total_opp_R$   = ('opportunity_cost', 'sum'),
        avg_roi        = ('roi_activation',   'mean'),
    )
    .round(2)
    .sort_values('total_opp_R$', ascending=False)
    .reset_index()
)
decision_table['action'] = decision_table['segment'].map(actions)
decision_table['avg_prob_30d'] = decision_table['avg_prob_30d'].map('{:.1%}'.format)
decision_table['avg_roi']      = decision_table['avg_roi'].map('{:.2f}x'.format)
decision_table['total_opp_R$'] = decision_table['total_opp_R$'].map('R${:,.0f}'.format)

print('══ FINAL DECISION TABLE ════════════════════════════════════════')
print(decision_table.to_string(index=False))
print(f'\nTotal 30d opportunity cost : R${customer_state["opportunity_cost"].sum():,.0f}')
print(f'Customers worth activating : {(customer_state["roi_activation"] > 0).sum():,} '
      f'({(customer_state["roi_activation"] > 0).mean():.1%} of base)')

---
## Conclusion — Answering the Central Question

> **Is customer segmentation a data problem or a marketing problem?**

It is a **false dichotomy** — and that's precisely the point.

| Without data | Without strategy |
|---|---|
| Marketing acts on intuition and arbitrary rules ("email everyone silent 60+ days") | The model produces probabilities that no one acts on |
| Budget is allocated linearly — treating a high-value at-risk customer the same as a dormant one | The Lorenz curve sits unused — you know the concentration but don't exploit it |
| Opportunity cost exists but is invisible | Opportunity cost is calculated but not converted into decisions |

**The Weibull model is the bridge:**
- It translates **past behavior** (inter-purchase intervals) into **present state** (days silent)
- It converts present state into **conditional probability** (the formula in Section 3)
- It converts probability into **expected value** and **opportunity cost** (Section 4)
- It converts opportunity cost into **ranked, actionable segments** (Section 5)

The answer is: **data solves the information problem. Marketing solves the execution problem. You need both, in that order.**

---
*Dataset: Olist Brazilian E-Commerce Public Dataset (Kaggle)*  
*Methods: Weibull MLE (SciPy), Survival Analysis, Conditional Probability, Lorenz/Gini*  
*Language: Python 3 · pandas · numpy · scipy · matplotlib*